# 6장 로지스틱 회귀: Python / TensorFlow / Keras / PyTorch 비교

원본 예제의 핵심은 다음과 같습니다.

- 입력 x: 공부 시간 = [2, 4, 6, 8, 10, 12, 14]
- 정답 y: 합격 여부 = [0, 0, 0, 1, 1, 1, 1]
- 모델: z = wx + b
- 확률: p = 1 / (1 + exp(-z))
- 손실함수: binary cross entropy
- 학습: SGD, learning rate = 0.01, 1000 epochs
- 테스트: 7시간 공부했을 때의 합격 확률

이 노트북에서는 같은 문제를 네 방식으로 구현합니다.

1. 순수 Python: 프레임워크 없이 미분식과 경사하강법을 직접 구현
2. TensorFlow: GradientTape를 이용한 저수준 구현
3. Keras: Sequential + Dense를 이용한 고수준 구현
4. PyTorch: nn.Linear + BCEWithLogitsLoss를 이용한 구현

비교를 쉽게 하기 위해 네 모델 모두 w = 0, b = 0에서 시작하고 같은 learning rate와 epoch 수를 사용합니다.

원본: https://github.com/taehojo/deeplearning_4th/blob/master/colab/ch06-colab.ipynb

## 공통 개념

로지스틱 회귀는 선형식 z = wx + b를 먼저 계산한 뒤 sigmoid 함수를 통과시켜 0과 1 사이의 확률을 만듭니다.

p = 1 / (1 + exp(-z))

정답이 y일 때 binary cross entropy는

loss = -[y log(p) + (1-y) log(1-p)]

입니다.

이 예제에서는 하나의 입력 x만 있으므로 학습해야 할 값은 가중치 w와 절편 b 두 개뿐입니다.

# 1. 순수 Python 구현

NumPy나 딥러닝 프레임워크 없이 sigmoid, loss, gradient를 직접 계산합니다.

In [ ]:
import math
import matplotlib.pyplot as plt

x_py = [2, 4, 6, 8, 10, 12, 14]
y_py = [0, 0, 0, 1, 1, 1, 1]

def sigmoid_py(z):
    # 큰 음수에서도 안정적으로 계산
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    ez = math.exp(z)
    return ez / (1.0 + ez)

w_py = 0.0
b_py = 0.0
lr = 0.01
epochs = 1000
n = len(x_py)
losses_py = []

for epoch in range(epochs):
    p = [sigmoid_py(w_py*x + b_py) for x in x_py]

    # binary cross entropy
    eps = 1e-12
    loss = -sum(
        y*math.log(max(prob, eps)) + (1-y)*math.log(max(1-prob, eps))
        for y, prob in zip(y_py, p)
    ) / n
    losses_py.append(loss)

    # BCE + sigmoid의 gradient
    dw = sum((prob-y)*x for x, y, prob in zip(x_py, y_py, p)) / n
    db = sum(prob-y for y, prob in zip(y_py, p)) / n

    w_py -= lr * dw
    b_py -= lr * db

print(f'w = {w_py:.6f}, b = {b_py:.6f}, final loss = {losses_py[-1]:.6f}')

hour = 7
pred_py = sigmoid_py(w_py*hour + b_py)
print(f'{hour}시간 공부할 경우 합격 예상 확률 = {pred_py*100:.1f}%')

In [ ]:
x_line = [i/10 for i in range(0, 161)]
y_line_py = [sigmoid_py(w_py*x + b_py) for x in x_line]

plt.figure(figsize=(7, 4.5))
plt.scatter(x_py, y_py, label='data')
plt.plot(x_line, y_line_py, label='Python')
plt.axvline(7, linestyle='--', alpha=0.5)
plt.xlabel('Study hours')
plt.ylabel('Pass probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

# 2. TensorFlow 저수준 구현

Keras의 Sequential/Dense를 사용하지 않고 tf.Variable, GradientTape, SGD로 직접 학습합니다.

In [ ]:
import tensorflow as tf

tf.random.set_seed(0)

x_tf = tf.constant([[2.], [4.], [6.], [8.], [10.], [12.], [14.]], dtype=tf.float32)
y_tf = tf.constant([[0.], [0.], [0.], [1.], [1.], [1.], [1.]], dtype=tf.float32)

w_tf = tf.Variable([[0.0]], dtype=tf.float32)
b_tf = tf.Variable([0.0], dtype=tf.float32)
optimizer_tf = tf.optimizers.SGD(learning_rate=0.01)
losses_tf = []

for epoch in range(1000):
    with tf.GradientTape() as tape:
        logits = tf.matmul(x_tf, w_tf) + b_tf
        loss = tf.reduce_mean(
            tf.nn.sigmoid_cross_entropy_with_logits(labels=y_tf, logits=logits)
        )

    gradients = tape.gradient(loss, [w_tf, b_tf])
    optimizer_tf.apply_gradients(zip(gradients, [w_tf, b_tf]))
    losses_tf.append(float(loss.numpy()))

print(f'w = {float(w_tf.numpy()[0,0]):.6f}, b = {float(b_tf.numpy()[0]):.6f}, final loss = {losses_tf[-1]:.6f}')

hour_tf = tf.constant([[7.0]], dtype=tf.float32)
pred_tf = float(tf.sigmoid(tf.matmul(hour_tf, w_tf) + b_tf).numpy()[0,0])
print(f'7시간 공부할 경우 합격 예상 확률 = {pred_tf*100:.1f}%')

In [ ]:
x_line_tf = tf.reshape(tf.linspace(0.0, 16.0, 161), (-1, 1))
y_line_tf = tf.sigmoid(tf.matmul(x_line_tf, w_tf) + b_tf).numpy().ravel()

plt.figure(figsize=(7, 4.5))
plt.scatter(x_py, y_py, label='data')
plt.plot(x_line, y_line_tf, label='TensorFlow')
plt.axvline(7, linestyle='--', alpha=0.5)
plt.xlabel('Study hours')
plt.ylabel('Pass probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

# 3. Keras 구현

원본과 가장 가까운 방식입니다. 하나의 Dense 뉴런에 sigmoid 활성화 함수를 사용합니다.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

x_keras = np.array([2, 4, 6, 8, 10, 12, 14], dtype=np.float32).reshape(-1, 1)
y_keras = np.array([0, 0, 0, 1, 1, 1, 1], dtype=np.float32)

model_keras = keras.Sequential([
    keras.Input(shape=(1,)),
    layers.Dense(
        1,
        activation='sigmoid',
        kernel_initializer='zeros',
        bias_initializer='zeros'
    )
])

model_keras.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01),
    loss='binary_crossentropy'
)

history_keras = model_keras.fit(x_keras, y_keras, epochs=1000, verbose=0)

w_keras, b_keras = model_keras.layers[0].get_weights()
print(f'w = {w_keras[0,0]:.6f}, b = {b_keras[0]:.6f}, final loss = {history_keras.history["loss"][-1]:.6f}')

pred_keras = float(model_keras.predict(np.array([[7.0]], dtype=np.float32), verbose=0)[0,0])
print(f'7시간 공부할 경우 합격 예상 확률 = {pred_keras*100:.1f}%')

In [ ]:
x_line_keras = np.linspace(0, 16, 161, dtype=np.float32).reshape(-1, 1)
y_line_keras = model_keras.predict(x_line_keras, verbose=0).ravel()

plt.figure(figsize=(7, 4.5))
plt.scatter(x_py, y_py, label='data')
plt.plot(x_line_keras.ravel(), y_line_keras, label='Keras')
plt.axvline(7, linestyle='--', alpha=0.5)
plt.xlabel('Study hours')
plt.ylabel('Pass probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

# 4. PyTorch 구현

nn.Linear(1, 1)이 z = wx + b를 계산하고 BCEWithLogitsLoss가 sigmoid와 binary cross entropy를 안정적으로 함께 계산합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

x_torch = torch.tensor([[2.], [4.], [6.], [8.], [10.], [12.], [14.]], dtype=torch.float32)
y_torch = torch.tensor([[0.], [0.], [0.], [1.], [1.], [1.], [1.]], dtype=torch.float32)

model_torch = nn.Linear(1, 1)
with torch.no_grad():
    model_torch.weight.zero_()
    model_torch.bias.zero_()

criterion = nn.BCEWithLogitsLoss()
optimizer_torch = optim.SGD(model_torch.parameters(), lr=0.01)
losses_torch = []

for epoch in range(1000):
    optimizer_torch.zero_grad()
    logits = model_torch(x_torch)
    loss = criterion(logits, y_torch)
    loss.backward()
    optimizer_torch.step()
    losses_torch.append(float(loss.item()))

w_torch = float(model_torch.weight.item())
b_torch = float(model_torch.bias.item())
print(f'w = {w_torch:.6f}, b = {b_torch:.6f}, final loss = {losses_torch[-1]:.6f}')

with torch.no_grad():
    pred_torch = float(torch.sigmoid(model_torch(torch.tensor([[7.0]]))).item())
print(f'7시간 공부할 경우 합격 예상 확률 = {pred_torch*100:.1f}%')

In [ ]:
x_line_torch = torch.linspace(0, 16, 161).reshape(-1, 1)
with torch.no_grad():
    y_line_torch = torch.sigmoid(model_torch(x_line_torch)).numpy().ravel()

plt.figure(figsize=(7, 4.5))
plt.scatter(x_py, y_py, label='data')
plt.plot(x_line_torch.numpy().ravel(), y_line_torch, label='PyTorch')
plt.axvline(7, linestyle='--', alpha=0.5)
plt.xlabel('Study hours')
plt.ylabel('Pass probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

# 5. 네 구현 결과 비교

동일한 초기값과 동일한 SGD 조건을 사용했으므로 네 방식의 w, b, loss와 7시간 예측값은 거의 같아야 합니다. 부동소수점 정밀도와 프레임워크 내부 계산 순서 때문에 아주 작은 차이는 생길 수 있습니다.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Method': ['Pure Python', 'TensorFlow', 'Keras', 'PyTorch'],
    'w': [w_py, float(w_tf.numpy()[0,0]), float(w_keras[0,0]), w_torch],
    'b': [b_py, float(b_tf.numpy()[0]), float(b_keras[0]), b_torch],
    'Final loss': [losses_py[-1], losses_tf[-1], history_keras.history['loss'][-1], losses_torch[-1]],
    'P(pass | 7 h)': [pred_py, pred_tf, pred_keras, pred_torch]
})
comparison

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(x_py, y_py, s=70, label='data')
plt.plot(x_line, y_line_py, label='Pure Python')
plt.plot(x_line, y_line_tf, '--', label='TensorFlow')
plt.plot(x_line_keras.ravel(), y_line_keras, ':', label='Keras')
plt.plot(x_line_torch.numpy().ravel(), y_line_torch, '-.', label='PyTorch')
plt.axvline(7, linestyle='--', alpha=0.35)
plt.xlabel('Study hours')
plt.ylabel('Pass probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

## 코드 구조 비교

- 순수 Python: w와 b의 gradient를 직접 계산하므로 로지스틱 회귀의 원리를 보기 가장 좋습니다.
- TensorFlow: 자동미분 GradientTape가 gradient를 계산하지만 모델 식은 직접 작성합니다.
- Keras: 모델, 손실함수, optimizer, 학습 반복을 가장 간단하게 작성할 수 있습니다.
- PyTorch: 모델과 optimizer를 객체로 정의하고 forward → loss → backward → step 순서를 명시적으로 수행합니다.

수업에서는 순수 Python → TensorFlow → Keras → PyTorch 순서로 실행하면 같은 수학이 프레임워크별로 어떻게 표현되는지 비교하기 좋습니다.